In [1]:
# Imports

from fastml.modules.cnn import load_qCNN
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import csv
from statistics import median

2026-07-27 18:05:12.606762: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-27 18:05:12.632547: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-27 18:05:12.632578: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-27 18:05:12.632602: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-27 18:05:12.638444: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-27 18:05:12.638823: I tensorflow/core/platform/cpu_feature_guard.cc:182] This Tens

In [ ]:
# Model paths and names

model1_path = '../Root/Models/model_512.keras'
model1_name = "ROOT"
model2_path = '../Original/Models/model_512.keras'
model2_name = "Original"

In [ ]:
# Other file paths

signal_path = 'Data/test_data/signal.parquet'
background_path = 'Data/test_data/background.parquet'

preprocessing_time_path = 'Data/preprocessing_time.csv'
training_times_path = 'Data/training_times.csv'
epoch_times_path = 'Data/epoch_times.csv'
batch_fetch_times_path = 'Data/batch_fetch_times.csv'

In [ ]:
# Signal + background data, and model generation

signal = ak.from_parquet(signal_path)
background = ak.from_parquet(background_path)
model1 = load_qCNN(model1_path)
model2 = load_qCNN(model2_path)

In [ ]:
# Generating predictions of testing data

model1_predictions = []
for dataset in [signal, background]:
    pred = model1.predict(
        np.array(ak.flatten(dataset.image)),
        batch_size=1024,
        verbose=0
    )
    
    pred = pred[:,0,0,0]
    counts = ak.num(dataset.seed_info)
    model1_predictions.append(ak.unflatten(pred, counts))
    
model1_predictions = {
    "signal" : model1_predictions[0],
    "background" : model1_predictions[1]
}
model2_predictions = []
for dataset in [signal, background]:
    pred = model2.predict(
        np.array(ak.flatten(dataset.image)),
        batch_size=1024,
        verbose=0
    )
    
    pred = pred[:,0,0,0]
    counts = ak.num(dataset.seed_info)
    model2_predictions.append(ak.unflatten(pred, counts))
    
model2_predictions = {
    "signal" : model2_predictions[0],
    "background" : model2_predictions[1]
}

In [ ]:
# ROC Curve

y_sig_1 = ak.flatten(model1_predictions['signal'])
y_bkg_1 = ak.flatten(model1_predictions['background'][:len(y_sig_1)])
y_1 = np.concatenate([np.ones_like(y_sig_1, dtype=int), np.zeros_like(y_bkg_1, dtype=int)])
s_1 = np.concatenate([y_sig_1, y_bkg_1])
fpr_1, tpr_1, _ = roc_curve(y_1, s_1)
A_1 = auc(fpr_1, tpr_1)

y_sig_2 = ak.flatten(model2_predictions['signal'])
y_bkg_2 = ak.flatten(model2_predictions['background'][:len(y_sig_2)])
y_2 = np.concatenate([np.ones_like(y_sig_2, dtype=int), np.zeros_like(y_bkg_2, dtype=int)])
s_2 = np.concatenate([y_sig_2, y_bkg_2])
fpr_2, tpr_2, _ = roc_curve(y_2, s_2)
A_2 = auc(fpr_2, tpr_2)

plt.figure()
ax = plt.axes()
ax.plot(fpr_1, tpr_1, color="blue", label=f"{model1_name} workflow, AUC={A_1:.3f}")
ax.plot(fpr_2, tpr_2, color="red", linestyle="--", label=f"{model2_name} workflow, AUC={A_2:.3f}")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC Curve")
plt.legend()

In [ ]:
# AU histogram

bin_edges = np.arange(0, 1, 0.05)

model1_sig, _ = np.histogram(ak.flatten(model1_predictions['signal']), bins=bin_edges, density=True)
model1_bkg, _ = np.histogram(ak.flatten(model1_predictions['background']), bins=bin_edges, density=True)

model2_sig, _ = np.histogram(ak.flatten(model2_predictions['signal']), bins=bin_edges, density=True)
model2_bkg, _ = np.histogram(ak.flatten(model2_predictions['background']), bins=bin_edges, density=True)

plt.figure(figsize=(10, 4))
plt.stairs(model1_sig, bin_edges, label=f"{model1_name} workflow signal", color="blue")
plt.stairs(model1_bkg, bin_edges, label=f"{model1_name} workflow background", linestyle="--", color="blue")
plt.stairs(model2_sig, bin_edges, label=f"{model2_name} workflow signal", color="red")
plt.stairs(model2_bkg, bin_edges, label=f"{model2_name} workflow background", linestyle="--", color="red")
plt.xlabel("Score")
plt.ylabel("AU")
plt.title("AU Histogram")
plt.legend()

In [ ]:
# Preprocessing time

workflows = None
# Alternatively, provide a list of workflows!

with open(preprocessing_time_path, newline="") as f:
    reader = csv.DictReader(f)
    for item in reader:
        times = item

if workflows is None:
    workflows = list(times)
else:
    missingHeaders = set(workflows) - times.keys()
    if missingHeaders:
        raise ValueError(f"These workflows aren't in {preprocessing_time_path}: \
                         {', '.join(missingHeaders)}")

plt.figure()
plt.bar(workflows, [float(times[workflow]) for workflow in workflows])
plt.xlabel("Workflow")
plt.ylabel("Preprocessing time (s)")
plt.title("Preprocessing Time Bar Chart")

In [ ]:
# Training times

workflows = None
# Alternatively, provide a list of workflows!

training_times_sum = {}
training_times_count = 0
with open(training_times_path, newline="") as f:
    reader = csv.DictReader(f)
    for d in reader:
        training_times_count += 1
        for workflow in d:
            training_times_sum.setdefault(workflow, 0)
            training_times_sum[workflow] += float(d[workflow])

if workflows is None:
    workflows = list(training_times_sum)
else:
    missingHeaders = set(workflows) - training_times_sum.keys()
    if missingHeaders:
        raise ValueError(f"These workflows aren't in {training_times_path}: \
                         {', '.join(missingHeaders)}")

plt.figure()
plt.bar(workflows, [float(training_times_sum[workflow]) for workflow in workflows])
plt.xlabel("Workflow")
plt.ylabel(f"{training_times_count} Training executions (s)")
plt.title("Training Time Bar Chart")

In [ ]:
# Epoch times

workflows = ["ROOT", "Original"]
# Alternatively, provide a list of workflows!

with open(epoch_times_path, newline="") as f:
    reader = csv.DictReader(f)
    headers = [h for h in reader.fieldnames if h not in ("run", "epoch")]
    values = {h: {} for h in headers}

    for d in reader:
        for h in headers:
            values[h].setdefault(d["epoch"], []).append(float(d[h]))

medians = {
    h: [median(epoch_times) for epoch_times in epochs.values()]
    for h, epochs in values.items()
}

if workflows is None:
    workflows = list(training_times_sum)
else:
    missingHeaders = set(workflows) - training_times_sum.keys()
    if missingHeaders:
        raise ValueError(f"These workflows aren't in {training_times_path}: \
                         {', '.join(missingHeaders)}")

plt.figure()
ax = plt.axes()
for workflow in workflows:
    ax.plot([i+1 for i in range(45)], medians[workflow], label=f"{workflow} workflow")
plt.xlabel("Epoch #")
plt.ylabel(f"Median epoch time (s)")
ax.set_yscale('log')
ax.set_yticks([0.02, 0.1, 1])
ax.set_yticklabels(["$2\\times10^{-2}$", "$10^{-1}$", "$10^0$"])
plt.title("Epoch times")
plt.legend()

In [ ]:
# Batch fetch times

workflows = ["Original", "ROOT"]
# Alternatively, provide None.
# NOTE: This graph by default only compares two workflows,
# as trying to put more becomes a little ugly.

batch_fetch_times = {}
with open(batch_fetch_times_path, newline="") as f:
    reader = csv.DictReader(f)
    for d in reader:
        for workflow in d:
            batch_fetch_times.setdefault(workflow, [])
            batch_fetch_times[workflow].append(float(d[workflow]))

if workflows is None:
    workflows = list(training_times_sum)
else:
    missingHeaders = set(workflows) - training_times_sum.keys()
    if missingHeaders:
        raise ValueError(f"These workflows aren't in {training_times_path}: \
                         {', '.join(missingHeaders)}")

plt.figure()
for i in range(len(workflows)):
    workflow = workflows[i]
    plt.hist(
        batch_fetch_times[workflow], 
        label=f"{workflow} workflow",
        bins=[i / 200000 for i in range(20, 40)],
        rwidth=1/(i+1)
    )

plt.xlabel("Batch fetch time (s)")
plt.ylabel(f"Count (sum to {len(batch_fetch_times[workflow])})")
plt.title("Batch Fetch Time Histogram")
plt.legend()